# PA Traffic Observations — all indicators

Loads the hourly `pa_traffic_observations.log-YYYYMMDDHH` files, parses the *Duplicate counts* section in each file (all session IP tuples), and builds **`source_data_matches`**: one row per deduplicated session tuple with `src_ip`, `dest_ip`, and related fields.

Each row is tagged with `source_hour` (from the filename) and `source_file`. The *Matching rows* section (observation hits only) is not used for indicator extraction.

For SOURCE vs TC alignment, unique indicators are dotted-decimal IPv4 addresses from `src_ip`, `dest_ip`, and masking columns (excluding `0.0.0.0`, IPv6, and malformed values).

**Run dates** are inferred from hourly log filenames in `LOG_DIR`; TC observations and workbook output follow those dates automatically.

In [27]:
import os
import re
import glob
from io import StringIO
from datetime import datetime

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

DATE_FMT = '%Y-%m-%d'
LOG_DIR = r'H:\Observation2.0\SourceData3Skull3Conf\pa_traffic_observations_3Skull50Conf'
OBS_DIR = r'Z:\HTOC\Data_Analytics\Data\OpDiv_Observations'
TC_INDEX_PATH = r'H:\ThreatConnect3skulls50conf.csv'
TC_SOURCE_OBS_XLSX = os.path.join(
    r'H:\HTOC\notebooks\Gap Observation 2.0',
    'TC_SOURCE_OBS_ANALYSIS.xlsx',
)
TC_SOURCE_OBS_CSV = os.path.join(
    r'H:\HTOC\notebooks\Gap Observation 2.0',
    'TC_SOURCE_OBS_ANALYSIS.csv',
)
files = sorted(glob.glob(os.path.join(LOG_DIR, 'pa_traffic_observations.log-*')))
print(f'Workbook: {TC_SOURCE_OBS_XLSX}')
print(f'Found {len(files)} files')
print('First:', os.path.basename(files[0]))
print('Last: ', os.path.basename(files[-1]))

Found 14 files
First: pa_traffic_observations.log-2026060300
Last:  pa_traffic_observations.log-2026060313


## Parser

Each file has a `Duplicate counts:` section (all session tuples as `combined_ips`) followed by `Matching rows:` (observation hits only). We parse the duplicate-count table, split `combined_ips` into `src_ip`, `dest_ip`, `masking_src_ip`, and `masking_dest_ip`, then tag rows with `source_hour` and `source_file`.

In [28]:
HOUR_RE = re.compile(r'log-(\d{10})$')
IP_COLS = ['src_ip', 'dest_ip', 'masking_src_ip', 'masking_dest_ip']
IPV4_RE = re.compile(r'^(?:\d{1,3}\.){3}\d{1,3}$')


def is_valid_ipv4_indicator(ip):
    """True for dotted-decimal IPv4 indicators; excludes 0.0.0.0 and IPv6."""
    ip = str(ip).strip()
    if ip in ('', '0.0.0.0', 'nan') or ':' in ip:
        return False
    if not IPV4_RE.match(ip):
        return False
    try:
        return all(0 <= int(o) <= 255 for o in ip.split('.'))
    except ValueError:
        return False


def hour_from_filename(path):
    m = HOUR_RE.search(os.path.basename(path))
    return datetime.strptime(m.group(1), '%Y%m%d%H') if m else None


def dates_from_log_files(paths):
    """Calendar dates (YYYY-MM-DD) covered by hourly log filenames."""
    dates = []
    for path in paths:
        hour = hour_from_filename(path)
        if hour is not None:
            dates.append(hour.strftime(DATE_FMT))
    return sorted(set(dates))


def obs_date_from_tc_file(path):
    """Parse calendar date from htoc_opdiv_obs_d*.csv filename (YYYYMMDD or YYYY-MM-DD)."""
    tail = os.path.splitext(os.path.basename(path).split('_d', 1)[-1])[0]
    for fmt, token in (('%Y%m%d', tail[:8]), ('%Y-%m-%d', tail[:10])):
        try:
            return datetime.strptime(token, fmt).strftime(DATE_FMT)
        except ValueError:
            continue
    return None


def _parse_fwf_section(part, section_name, path):
    lines = [ln for ln in part.splitlines() if ln.strip()]
    if not lines:
        return pd.DataFrame()
    buf = StringIO('\n'.join(lines))
    try:
        df = pd.read_fwf(buf)
        if str(df.columns[0]).lower().startswith('unnamed'):
            df = df.drop(columns=df.columns[0])
        return df
    except Exception as e:
        print(f'  ! could not parse {section_name} in {os.path.basename(path)}: {e}')
        return pd.DataFrame()


def parse_file(path):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        text = f.read()

    if 'Duplicate counts:' not in text:
        return pd.DataFrame()

    _, rest = text.split('Duplicate counts:', 1)
    dup_part = rest.split('Matching rows:', 1)[0]
    dup_df = _parse_fwf_section(dup_part, 'duplicate counts', path)
    if dup_df.empty or 'combined_ips' not in dup_df.columns:
        return dup_df

    parts = dup_df['combined_ips'].astype(str).str.split(',', expand=True)
    n = parts.shape[1]
    col_map = {0: 'src_ip', 1: 'dest_ip', 2: 'masking_src_ip', 3: 'masking_dest_ip'}
    for i in range(min(n, 4)):
        dup_df[col_map[i]] = parts[i].astype(str).str.strip()
    return dup_df


def indicators_from_df(df):
    """Unique dotted-decimal IPv4 indicators from all address columns."""
    cols = [c for c in IP_COLS if c in df.columns]
    if not cols:
        return pd.Series(dtype=str)
    ips = df[cols].astype(str).apply(lambda s: s.str.strip())
    stacked = ips.stack(future_stack=True).drop_duplicates()
    return stacked[stacked.map(is_valid_ipv4_indicator)]


_m = parse_file(files[0])
print('Sample file:', os.path.basename(files[0]))
print('  session tuples:', len(_m))
print('  unique indicators:', len(indicators_from_df(_m)))
_m

Sample file: pa_traffic_observations.log-2026060300
  session tuples: 9728
  unique indicators: 12742


,combined_ips,count,src_ip,dest_ip,masking_src_ip,masking_dest_ip
0,"1.140.118.155,158.111.224.97,0.0.0.0,0.0.0.0",1,1.140.118.155,158.111.224.97,0.0.0.0,0.0.0.0
1,"1.178.192.250,156.40.212.210,0.0.0.0,0.0.0.0",1,1.178.192.250,156.40.212.210,0.0.0.0,0.0.0.0
2,"10.128.55.48,18.254.8.189,0.0.0.0,0.0.0.0",1,10.128.55.48,18.254.8.189,0.0.0.0,0.0.0.0
3,"10.130.104.200,18.254.255.124,0.0.0.0,0.0.0.0",1,10.130.104.200,18.254.255.124,0.0.0.0,0.0.0.0
4,"100.15.85.91,158.72.66.137,0.0.0.0,0.0.0.0",1,100.15.85.91,158.72.66.137,0.0.0.0,0.0.0.0
...,...,...,...,...,...,...
9723,"96.63.183.43,150.148.227.222,0.0.0.0,0.0.0.0",1,96.63.183.43,150.148.227.222,0.0.0.0,0.0.0.0
9724,"97.221.93.182,150.148.18.66,0.0.0.0,0.0.0.0",1,97.221.93.182,150.148.18.66,0.0.0.0,0.0.0.0
9725,"98.19.128.219,158.111.210.149,0.0.0.0,0.0.0.0",1,98.19.128.219,158.111.210.149,0.0.0.0,0.0.0.0
9726,"98.8.35.135,165.112.4.230,0.0.0.0,0.0.0.0",1,98.8.35.135,165.112.4.230,0.0.0.0,0.0.0.0


In [29]:
_m.columns.tolist()

['combined_ips',
 'count',
 'src_ip',
 'dest_ip',
 'masking_src_ip',
 'masking_dest_ip']

## Load all files

Loops through every hourly file and stacks all duplicate-count session tuples. Each row gets `source_hour` (a real `Timestamp`) and `source_file`.

In [30]:
match_frames = []

for i, path in enumerate(files, 1):
    hour = hour_from_filename(path)
    fname = os.path.basename(path)
    m = parse_file(path)
    if not m.empty:
        m.insert(0, 'source_hour', hour)
        m.insert(1, 'source_file', fname)
        match_frames.append(m)
    if i % 25 == 0 or i == len(files):
        print(f'  parsed {i}/{len(files)}')

source_data_matches = pd.concat(match_frames, ignore_index=True) if match_frames else pd.DataFrame()

print()
print(f'source_data_matches: {len(source_data_matches):,} rows  x  {source_data_matches.shape[1]} cols')

  parsed 14/14

source_data_matches: 133,041 rows  x  8 cols


In [31]:
pd.DataFrame({'indicator': indicators_from_df(source_data_matches).sort_values().values})

,indicator
0,01.123.213.105
1,02.152.254.230
2,02.180.158.204
3,02.216.224.160
4,03.163.233.254
...,...
128685,99.84.41.93
128686,99.92.21.66
128687,99.92.21.86
128688,99.95.61.176


In [32]:
# Load TC observations for every calendar day in the current SOURCE log batch
RUN_DATES = dates_from_log_files(files)
print(f'Run dates from SOURCE logs: {RUN_DATES}')

opdiv_obs_files = sorted(glob.glob(os.path.join(OBS_DIR, 'htoc_opdiv_obs_d*.csv')))
run_dates_set = set(RUN_DATES)

filtered_files = [
    f for f in opdiv_obs_files
    if obs_date_from_tc_file(f) in run_dates_set
]
print(f'TC observation files matched: {len(filtered_files)}')

if not filtered_files:
    raise FileNotFoundError(
        f'No TC observation files in {OBS_DIR} for run dates {RUN_DATES}'
    )

TC_obs_df = (
    pd.concat([pd.read_csv(f) for f in filtered_files], ignore_index=True)
    .drop_duplicates(subset=['indicator', 'obs_date'])
)
TC_obs_df

Run dates from SOURCE logs: ['2026-06-03']
TC observation files matched: 1


,indicator,API_UserName,obs_date,OpDiv,indicator_key,observations,curr_date
0,1.83.125.95,35664937356778434955,2026-06-03,HRSA,1.83.125.95_HRSA,24,2026-06-03
1,101.132.182.180,35664937356778434955,2026-06-03,HRSA,101.132.182.180_HRSA,20,2026-06-03
2,101.47.8.187,35664937356778434955,2026-06-03,HRSA,101.47.8.187_HRSA,48,2026-06-03
3,102.219.210.202,35664937356778434955,2026-06-03,HRSA,102.219.210.202_HRSA,3,2026-06-03
4,102.89.82.4,74198686107399967946,2026-06-03,VA,102.89.82.4_VA,6,2026-06-03
...,...,...,...,...,...,...,...
4358,whatistheurl.com,20790633968691748718,2026-06-03,DHA,whatistheurl.com_DHA,19,2026-06-03
4359,whistlingmorans.com,50189120947314147395,2026-06-03,NIH,whistlingmorans.com_NIH,3,2026-06-03
4360,wvmetronews.com,20790633968691748718,2026-06-03,DHA,wvmetronews.com_DHA,405,2026-06-03
4361,www.deepseek.com,20790633968691748718,2026-06-03,DHA,www.deepseek.com_DHA,9,2026-06-03


## SOURCE vs TC — same-day alignment

For each calendar day and each indicator (IP):

| Scenario | SOURCE (log) | TC |
|----------|--------------|-----|
| `aligned_same_day` | X | X — observed the same day in both |
| `source_only_not_in_tc` | X | — in log that day, not in TC that day |
| `tc_only_not_in_source` | — | X — in TC that day, not in log that day |

Output **`source_tc_presence`**: one row per indicator in **`H:\ThreatConnect3skulls50conf.csv`** and per compare date, with `X` in `source` and/or `ThreatConnect` when that IP was seen that day in logs or TC observations.

Writes **`TC_SOURCE_OBS_ANALYSIS.xlsx`** in `Gap Observation 2.0/`:

- Sheet **`All`** — all dates combined (rebuilt each run)
- One sheet per date (`YYYY-MM-DD`) with metric counts at the top, then indicators
- **Existing date sheets are kept**; only sheets for **new** run dates are added

In [38]:
# SOURCE vs TC — same-day indicator alignment
# For each calendar day: was each IP seen in the PA log (SOURCE) also recorded in TC that day, and vice versa?

# RUN_DATES comes from cell 8 (SOURCE log filenames); re-use if already set
if 'RUN_DATES' not in globals():
    RUN_DATES = dates_from_log_files(files)

_OUTPUT_DIR = r'H:\HTOC\notebooks\Gap Observation 2.0'
TC_SOURCE_OBS_XLSX = os.path.join(_OUTPUT_DIR, 'TC_SOURCE_OBS_ANALYSIS.xlsx')
TC_SOURCE_OBS_CSV = os.path.join(_OUTPUT_DIR, 'TC_SOURCE_OBS_ANALYSIS.csv')
if 'TC_INDEX_PATH' not in globals():
    TC_INDEX_PATH = r'H:\ThreatConnect3skulls50conf.csv'


def _normalize_ip(s):
    return s.astype(str).str.strip()


def _indicator_column(df, path=''):
    """Resolve indicator/IP column (handles BOM headers and alternate names)."""
    cols = [str(c).strip().lstrip('\ufeff') for c in df.columns]
    df = df.rename(columns=dict(zip(df.columns, cols)))
    for name in ('indicator', 'ip', 'IP', 'address'):
        if name in df.columns:
            return name
    if len(df.columns) == 1:
        return df.columns[0]
    raise KeyError(f"No indicator column in {path}; columns: {list(df.columns)}")


def _load_indicators_from_csv(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    col = _indicator_column(df, path)
    return _normalize_ip(df[col])


tc_index_indicators = {
    ip for ip in _load_indicators_from_csv(TC_INDEX_PATH).dropna()
    if is_valid_ipv4_indicator(ip)
}


def _source_obs_dates(df):
    """Calendar date for each log row (session date, else log-file hour)."""
    ts = pd.Series(pd.NaT, index=df.index)
    unnamed = [c for c in df.columns if str(c).lower().startswith('unnamed')]
    if unnamed and 'timestamp' in df.columns:
        combined = (
            df[unnamed[0]].astype(str).str.strip()
            + ' '
            + df['timestamp'].astype(str).str.strip()
        )
        ts = pd.to_datetime(combined, format='%Y/%m/%d %H:%M:%S', errors='coerce')
    elif 'timestamp' in df.columns:
        ts = pd.to_datetime(df['timestamp'], errors='coerce')
    if ts.isna().all() and 'source_hour' in df.columns:
        ts = pd.to_datetime(df['source_hour'], errors='coerce')
    return ts.dt.strftime(DATE_FMT)


# Unique (date, indicator) seen in SOURCE logs (all IPs in each session tuple)
_ip_cols = [c for c in IP_COLS if c in source_data_matches.columns]
_source_long = source_data_matches.assign(
    obs_date=_source_obs_dates(source_data_matches),
).melt(id_vars=['obs_date'], value_vars=_ip_cols, value_name='indicator')

source_daily = (
    _source_long.assign(indicator=_normalize_ip(_source_long['indicator']))
    .dropna(subset=['obs_date', 'indicator'])
    .loc[lambda d: d['indicator'].map(is_valid_ipv4_indicator)]
    .drop_duplicates(subset=['obs_date', 'indicator'])
    [['obs_date', 'indicator']]
    .assign(in_source=True)
)

# Unique (date, indicator) recorded in TC
_tc_ind_col = _indicator_column(TC_obs_df, 'TC_obs_df')
tc_daily = (
    TC_obs_df.assign(
        obs_date=pd.to_datetime(TC_obs_df['obs_date'], errors='coerce').dt.strftime(DATE_FMT),
        indicator=_normalize_ip(TC_obs_df[_tc_ind_col]),
    )
    .dropna(subset=['obs_date', 'indicator'])
    .query('indicator != ""')
    .drop_duplicates(subset=['obs_date', 'indicator'])
    [['obs_date', 'indicator']]
    .assign(in_tc=True)
)

PRESENT = 'X'

# Every index indicator x every run date in this SOURCE log batch
_compare_dates = sorted(
    set(RUN_DATES)
    | set(source_daily['obs_date'].dropna())
    | set(tc_daily['obs_date'].dropna())
)
_compare_dates = [d for d in _compare_dates if d and str(d) != 'NaT']
print(f'Building presence for {len(_compare_dates)} day(s): {_compare_dates}')
_index_base = pd.DataFrame({'indicator': sorted(tc_index_indicators)})
_dates_base = pd.DataFrame({'obs_date': _compare_dates})
_presence_base = _index_base.assign(_key=1).merge(_dates_base.assign(_key=1), on='_key').drop(columns='_key')

_merged = (
    _presence_base
    .merge(source_daily, on=['obs_date', 'indicator'], how='left')
    .merge(tc_daily, on=['obs_date', 'indicator'], how='left')
)
_merged['in_source'] = _merged['in_source'].eq(True)
_merged['in_tc'] = _merged['in_tc'].eq(True)

# Final output: indicator, date, presence markers (X = seen that day)
source_tc_presence = (
    _merged
    .assign(
        source=lambda d: d['in_source'].map({True: PRESENT, False: ''}),
        ThreatConnect=lambda d: d['in_tc'].map({True: PRESENT, False: ''}),
    )
    .rename(columns={'obs_date': 'date'})
    [['indicator', 'date', 'source', 'ThreatConnect']]
    .sort_values(['date', 'indicator'])
    .reset_index(drop=True)
)

def _presence_metrics(df):
    """Counts for SOURCE only, TC only, both, neither (X = seen that day)."""
    src = df['source'].astype(str).str.strip().eq('X')
    tc = df['ThreatConnect'].astype(str).str.strip().eq('X')
    return {
        'SOURCE only': int((src & ~tc).sum()),
        'ThreatConnect only': int((~src & tc).sum()),
        'Both': int((src & tc).sum()),
        'Neither': int((~src & ~tc).sum()),
        'Total indicators': len(df),
    }


def _data_from_date_sheet(df):
    """Strip metrics header rows when reloading an existing workbook sheet."""
    if df is None or df.empty:
        return df
    cols = {str(c).lower(): c for c in df.columns}
    ind_col = cols.get('indicator', df.columns[0])
    ind = df[ind_col].astype(str).str.strip()
    if '--- metrics ---' not in ind.str.lower().values:
        return df.reset_index(drop=True)
    ip_mask = ind.map(is_valid_ipv4_indicator)
    return df.loc[ip_mask].reset_index(drop=True)


def _sheet_with_metrics(day_df):
    """Metrics block at top; counts in `source` column, then indicator rows."""
    data = day_df[['indicator', 'source', 'ThreatConnect']].copy()
    m = _presence_metrics(data)
    summary = pd.DataFrame(
        [
            {'indicator': '--- Metrics ---', 'source': '', 'ThreatConnect': ''},
            {'indicator': 'SOURCE only', 'source': m['SOURCE only'], 'ThreatConnect': ''},
            {'indicator': 'ThreatConnect only', 'source': m['ThreatConnect only'], 'ThreatConnect': ''},
            {'indicator': 'Both', 'source': m['Both'], 'ThreatConnect': ''},
            {'indicator': 'Neither', 'source': m['Neither'], 'ThreatConnect': ''},
            {'indicator': 'Total indicators', 'source': m['Total indicators'], 'ThreatConnect': ''},
            {'indicator': '', 'source': '', 'ThreatConnect': ''},
        ]
    )
    return pd.concat([summary, data], ignore_index=True)


def _load_existing_date_sheets(xlsx_path):
    if not os.path.exists(xlsx_path):
        return {}
    sheets = pd.read_excel(xlsx_path, sheet_name=None)
    return {
        str(name): _data_from_date_sheet(df)
        for name, df in sheets.items()
        if str(name) != 'All'
    }


def write_tc_source_obs_workbook(run_presence, xlsx_path, csv_path):
    """Add sheets for new dates only; keep existing date sheets unchanged."""
    date_sheets = _load_existing_date_sheets(xlsx_path)
    existing_dates = set(date_sheets.keys())
    run_metrics = {}
    added_dates = []
    skipped_dates = []

    for obs_date, day_df in run_presence.groupby('date', sort=True):
        date_str = str(obs_date)
        raw = day_df.drop(columns=['date']).reset_index(drop=True)
        run_metrics[date_str] = _presence_metrics(raw)
        if date_str in existing_dates:
            skipped_dates.append(date_str)
            continue
        date_sheets[date_str] = raw
        added_dates.append(date_str)

    all_parts = [
        date_sheets[obs_date].assign(date=obs_date)
        for obs_date in sorted(date_sheets.keys())
    ]
    all_presence = (
        pd.concat(all_parts, ignore_index=True)
        [['indicator', 'date', 'source', 'ThreatConnect']]
        .sort_values(['date', 'indicator'])
        .reset_index(drop=True)
    )

    all_presence.to_csv(csv_path, index=False)
    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
        all_presence.to_excel(writer, sheet_name='All', index=False)
        for obs_date in sorted(date_sheets.keys()):
            _sheet_with_metrics(date_sheets[obs_date]).to_excel(
                writer, sheet_name=obs_date, index=False
            )

    return all_presence, sorted(date_sheets.keys()), run_metrics, added_dates, skipped_dates


tc_source_obs_all, workbook_dates, run_metrics, added_dates, skipped_dates = (
    write_tc_source_obs_workbook(
        source_tc_presence, TC_SOURCE_OBS_XLSX, TC_SOURCE_OBS_CSV
    )
)

print(
    f'This run: {len(source_tc_presence):,} indicator-day rows '
    f'({len(tc_index_indicators):,} index indicators x {source_tc_presence["date"].nunique()} day(s))'
)
for obs_date, m in run_metrics.items():
    print(
        f'  {obs_date}: SOURCE only={m["SOURCE only"]:,} | TC only={m["ThreatConnect only"]:,} | '
        f'Both={m["Both"]:,} | Neither={m["Neither"]:,}'
    )
if added_dates:
    print(f'Added sheet(s): {added_dates}')
if skipped_dates:
    print(f'Kept existing sheet(s) (not replaced): {skipped_dates}')
print(f'Workbook has {len(workbook_dates)} date sheet(s): {workbook_dates}')
print(f'Wrote {TC_SOURCE_OBS_CSV}')
print(f'Wrote {TC_SOURCE_OBS_XLSX}')
source_tc_presence

Building presence for 1 day(s): ['2026-06-03']
This run: 3,426 indicator-day rows (3,426 index indicators x 1 day(s))
  2026-06-03: SOURCE only=0 | TC only=527 | Both=354 | Neither=2,545
Added sheet(s): ['2026-06-03']
Workbook has 1 date sheet(s): ['2026-06-03']
Wrote H:\HTOC\notebooks\Gap Observation 2.0\TC_SOURCE_OBS_ANALYSIS.csv
Wrote H:\HTOC\notebooks\Gap Observation 2.0\TC_SOURCE_OBS_ANALYSIS.xlsx


,indicator,date,source,ThreatConnect
0,1.13.188.57,2026-06-03,,
1,1.20.169.90,2026-06-03,,
2,1.234.83.26,2026-06-03,,
3,1.54.41.9,2026-06-03,,
4,101.168.57.163,2026-06-03,,
...,...,...,...,...
3421,98.159.226.75,2026-06-03,,
3422,98.159.226.82,2026-06-03,,
3423,98.170.186.3,2026-06-03,,
3424,98.20.106.25,2026-06-03,,
